# Pronóstico de utilización de materiales con Random Forest (RF30)

**Proyecto académico — Maestría en Inteligencia de Negocios y Ciencia de Datos (2026)**  
**Autor:** Miguel Angel Tapia Caluqui

Este notebook documenta de manera ordenada el proceso de análisis: preparación de datos, control de calidad, consolidación de losas físicas, análisis exploratorio, ingeniería de variables, entrenamiento RF30, comparación contra modelos base, validación temporal y proyección a 12 meses.

> **Advertencia de procedencia:** esta versión pública funciona con datos **simulados**. No incluye las reglas de negocio ni la base operativa original. Los resultados generados aquí **no son** los resultados de la tesis. La metodología descrita en [`docs/metodologia.md`](../docs/metodologia.md) delimita las diferencias.

## 0. Preparar el entorno

Desde la raíz del repositorio, ejecuta `pip install -e ".[dev]"` para instalar dependencias. El notebook importa los módulos del directorio `src/`.

**En Google Colab**: primero sube y descomprime el ZIP completo en `/content/Pronostico_Demanda_Materiales_RF30/`, o clona tu repositorio una vez publicado. En Colab, puedes instalar dependencias con `!pip install -r /content/Pronostico_Demanda_Materiales_RF30/requirements.txt`. No subas archivos privados a un entorno compartido.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Ubicación local, Jupyter o Google Colab (con repositorio completo).
RAIZ = Path.cwd()
if (RAIZ / "src").exists():
    pass
elif (RAIZ.parent / "src").exists():
    RAIZ = RAIZ.parent
elif Path("/content/Pronostico_Demanda_Materiales_RF30/src").exists():
    RAIZ = Path("/content/Pronostico_Demanda_Materiales_RF30")
else:
    raise FileNotFoundError(
        "No se encontró src/. Abre el notebook desde el repositorio completo, "
        "o descomprime la carpeta en /content/Pronostico_Demanda_Materiales_RF30."
    )
sys.path.insert(0, str(RAIZ / "src"))
print("Repositorio:", RAIZ)

## 1. Definición del problema y configuración

La variable de respuesta es el **número de losas físicas diferentes utilizadas por material y mes**. Se mantienen separadas la utilización, la solicitud y una aproximación descriptiva de material FULL nuevo.

El modelo Random Forest tiene 30 árboles, profundidad máxima 8, mínimo de 2 observaciones por hoja y selección `sqrt` de variables, como consta en la metodología de la tesis. El corte determina qué meses se consideran completos.

In [ ]:
from material_forecasting.config import Config

config = Config(fecha_corte="2026-11-01", top_n=30, semilla=42)
print(config)

## 2. Lectura y auditoría de datos

Los registros publicados son artificiales. Cada fila representa un evento de asignación a un trabajo; se conserva un identificador de losa física base para evitar duplicar remanentes. Las fechas inválidas y el mes parcial/futuro se separan para revisión, en lugar de tratarlos automáticamente como consumo cero.

In [ ]:
from material_forecasting.ejemplo import generar_csv
from material_forecasting.datos import leer_eventos, preparar_eventos

archivo = RAIZ / "data/simulada/eventos_simulados.csv"
if not archivo.exists():
    generar_csv(archivo, semilla=config.semilla)

entrada = leer_eventos(archivo)
historico, revision = preparar_eventos(entrada, config.fecha_corte)
print(f"Registros leídos: {len(entrada):,}")
print(f"Registros históricos válidos: {len(historico):,}")
print(f"Registros en revisión: {len(revision):,}")
display(historico.head())

## 3. Consolidación mensual y selección del portafolio

La agregación se realiza por material-mes contando los `slab_id` físicos distintos. Se genera un panel mensual con meses sin utilización como cero, siempre que el periodo esté efectivamente cubierto. El Top 30 se define a partir de los últimos 12 meses completos.

In [ ]:
from material_forecasting.datos import panel_mensual
from material_forecasting.modelos import panel_a_series

mes_final = (pd.Timestamp(config.fecha_corte).to_period("M") - 1).to_timestamp()
panel = panel_mensual(historico, config.top_n, inicio=config.fecha_inicio, fin=mes_final)
meses, series = panel_a_series(panel)
print(f"Materiales priorizados: {len(series)}")
print(f"Meses completos: {len(meses)}")
display(panel.head(10))

## 4. Análisis exploratorio

Es necesario identificar tendencia, estacionalidad, heterogeneidad e intermitencia antes de elegir un algoritmo. La relación entre meses consecutivos permite examinar dependencia temporal, pero no demuestra causalidad.

In [ ]:
import matplotlib.pyplot as plt

mensual = panel.groupby("mes")["utilizacion"].sum()
ax = mensual.plot(figsize=(9, 3.8), marker="o", markersize=3, grid=False)
ax.set_xlabel("Mes")
ax.set_ylabel("Losas físicas utilizadas")
plt.tight_layout()
plt.show()

## 5. Ingeniería de variables y Random Forest

Se utilizan rezagos de consumo, estadísticas móviles, tendencias, tasas de actividad, uso de material FULL/remnant y calendario. El entrenamiento supervisado combina varios materiales. La variable de horizonte permite al bosque aprender pronósticos para los 12 meses futuros.

In [ ]:
from material_forecasting.modelos import ejemplos_entrenamiento, entrenar_rf30

ejemplos = ejemplos_entrenamiento(meses, series, config.meses_pronostico)
modelo_global = entrenar_rf30(ejemplos, config)
print("Filas supervisadas:", len(ejemplos))
print("Árboles:", modelo_global.modelo.n_estimators if modelo_global else "Sin suficiente historia")
display(ejemplos.head(3))

## 6. Evaluación mediante backtesting de origen móvil

Se simulan decisiones pasadas. En cada origen se entrena un nuevo Random Forest usando **exclusivamente objetivos observados hasta ese momento** y se compara con métodos clásicos: Naive, promedios móviles, SES, estacional ingenuo y Croston SBA. Se evalúan 1, 3, 6 y 12 meses puntuales.

Se elige un método por material y horizonte usando los primeros orígenes, y el conjunto posterior se reserva para una evaluación temporal externa. No se mezclan al azar los meses.

In [ ]:
from material_forecasting.evaluacion import backtesting, evaluar_adaptativo, metricas_puntuales

backtest = backtesting(meses, series, config)
adaptativo_prueba, seleccion_calibracion = evaluar_adaptativo(backtest, config)
prueba = backtest[backtest["origen"].isin(adaptativo_prueba["origen"].unique())]
comparacion = metricas_puntuales(pd.concat([prueba, adaptativo_prueba]), config.horizontes)
print("Orígenes de evaluación:", backtest["origen"].nunique())
display(comparacion.round(2))

## 7. Diferencia entre horizonte puntual y acumulado

El **horizonte puntual h=3** mide sólo el error en el tercer mes. En cambio, el **horizonte acumulado 3M** compara la suma de los siguientes tres meses con el total observado de esa ventana.

- WAPE = suma del error absoluto / suma observada × 100.
- MAE = error absoluto medio en losas.
- RMSE = raíz del error cuadrático medio en losas.
- Bias = diferencia neta pronóstico − real, como % del volumen real.

Un WAPE de 30% no equivale a «70% de exactitud».

In [ ]:
from material_forecasting.evaluacion import metricas_acumuladas, wape

acumulado = metricas_acumuladas(pd.concat([prueba, adaptativo_prueba]), config.horizontes)
resumen_acumulado = []
for (metodo, h), grupo in acumulado.groupby(["metodo", "horizonte"]):
    errores = [wape(g.real, g.pronostico) for _, g in grupo.groupby("material")]
    resumen_acumulado.append({"metodo": metodo, "horizonte": h, "wape_mediano": pd.Series(errores).median()})
display(pd.DataFrame(resumen_acumulado).sort_values(["horizonte", "wape_mediano"]).round(2))

## 8. Pronóstico final de 12 meses

Se vuelve a entrenar con todo el historial completo; el método seleccionado para cada material y horizonte puede ser Random Forest o un baseline si el backtesting fue más favorable. Esta es una señal de **utilización física esperada**, no una orden de compra ni un cálculo neto de inventario.

In [ ]:
from material_forecasting.evaluacion import pronostico_final

pronostico = pronostico_final(meses, series, backtest, config)
display(pronostico.head(15))

## 9. Visualizaciones académicas

Se exportan siete gráficos en alta resolución. La concentración por constructora en esta versión es **histórica** (no corresponde al módulo privado de pronóstico por cuenta). El proxy FULL es orientativo y no considera inventario, órdenes abiertas ni tiempos de entrega.

In [ ]:
from material_forecasting.graficos import generar_figuras

carpeta_figuras = RAIZ / "outputs/ejecucion_notebook/figuras"
figuras = generar_figuras(panel, pronostico, backtest, adaptativo_prueba, historico, carpeta_figuras)
print("Figuras guardadas:")
for nombre in figuras:
    print("-", nombre + ".png")

## 10. Exportación y conclusiones

El sistema exporta los resultados a CSV y mantiene los datos internos fuera del repositorio. La utilidad para compras consiste en identificar referencias con mayor previsión, reconocer el sesgo y discutir excepciones. La implementación operativa exigiría integrar inventario, órdenes abiertas, costo, lead time y material comprometido.

**Limitación principal:** el notebook de demostración no reproduce las métricas numéricas de la tesis; los datos y ciertas reglas del motor original son privados. No deben copiarse porcentajes de este ejemplo en el manuscrito académico.

In [ ]:
carpeta_salidas = RAIZ / "outputs/ejecucion_notebook"
carpeta_salidas.mkdir(parents=True, exist_ok=True)
for nombre, tabla in {"panel_mensual": panel, "backtesting": backtest,
                      "comparacion_wape": comparacion, "pronosticos": pronostico}.items():
    tabla.to_csv(carpeta_salidas / f"{nombre}.csv", index=False, encoding="utf-8-sig")
print("Resultados guardados en:", carpeta_salidas)